In [1]:
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("openrouter_api_key"), (
    "Missing OPENROUTER_API_KEY -- copy .env.example to .env and fill in your key."
)

from langchain.mcp import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain.mcp import MCPAdapter
from langchain.agents import create_agent
from fastmcp.client import Client
from fastmcp.client.transports import PythonStdioTransport

print("Environment ready.")

C:\Users\mohsinurrahman\AppData\Local\Temp\ipykernel_34916\316114197.py:11: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import LangChainBetaWarning


Environment ready.


In [3]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """
    Check showtimes for a given movie title.
    """
    fake_showtimes = {
        "Inception": "10:00 AM, 1:00 PM, 4:00 PM, 7:00 PM",
        "The Matrix": "11:00 AM, 2:00 PM, 5:00 PM, 8:00 PM",
        "Interstellar": "12:00 PM, 3:00 PM, 6:00 PM, 9:00 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes available for this movie.")


if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_mcp_server.py


Connect to each server separately

In [5]:
all_tools=[]
server_status={}
#--local cinebot 

try:
    cinebot_transport=PythonStdioTransport(
        script_path=Path("cinebot_mcp_server.py"),
        log_file=Path("cinebot_mcp_server.log"),

    )

    async with MCPAdapter(cinebot_transport) as adapter:
        tools=await adapter.list_tools()
        all_tools.extend(tools)
        server_status["cinebot(local)"] = f"OK--{[t.name for t in tools]}"


except Exception as e:
    server_status["cinebot(local)"] = f"failed: {type(e).__name__}"

for name,status in server_status.items():
    print(f"{name}: {status}")
     

cinebot(local): OK--['check_showtimes']


#Connect to free public server

In [6]:
#ctx7sk-7d7fbc49-94df-4bf5-8819-a4519138aabe

# -- 2. Context7 (free, public, documented to work keyless at lower rate limits) --
# https://mcp.context7.com/mcp -- pass CONTEXT7_API_KEY as a bearer token if you have one, for
# higher rate limits; works without one for a quick demo.
context7_key = os.environ.get("ctx7sk-7d7fbc49-94df-4bf5-8819-a4519138aabe")
try:
    context7_client = (
        Client("https://mcp.context7.com/mcp", auth=context7_key)
        if context7_key
        else Client("https://mcp.context7.com/mcp")
    )
    async with MCPAdapter(context7_client) as adapter:
        tools = await adapter.list_tools()
        all_tools.extend(tools)
        server_status["context7 (remote, free)"] = f"OK -- {[t.name for t in tools]}"
except Exception as e:
    server_status["context7 (remote, free)"] = f"FAILED -- {type(e).__name__}: {e}"

print(f"context7 (remote, free): {server_status['context7 (remote, free)']}")

context7 (remote, free): OK -- ['resolve-library-id', 'query-docs']


Connect o vercel time track mcp server

In [7]:
# -- 3. Your own remote server --
# Swap this URL for your own deployment. No API key assumed here -- add one via `auth=` the same
# way as the Context7 cell above if your server needs it.
TIMETRACK_URL = "https://time-track-mcp-server-mcpg-uys.vercel.app/mcp"

try:
    async with MCPAdapter(TIMETRACK_URL) as adapter:
        tools = await adapter.list_tools()
        all_tools.extend(tools)
        server_status["time-track (your remote server)"] = f"OK -- {[t.name for t in tools]}"
except Exception as e:
    server_status["time-track (your remote server)"] = f"FAILED -- {type(e).__name__}: {e}"

print(f"time-track (your remote server): {server_status['time-track (your remote server)']}")

print("\n--- Fleet summary ---")
for name, status in server_status.items():
    print(f"{name}: {status}")
print(f"\nTotal tools available to the agent: {len(all_tools)}")

time-track (your remote server): OK -- ['log_time', 'get_timesheet', 'get_project_summary', 'list_projects']

--- Fleet summary ---
cinebot(local): OK--['check_showtimes']
context7 (remote, free): OK -- ['resolve-library-id', 'query-docs']
time-track (your remote server): OK -- ['log_time', 'get_timesheet', 'get_project_summary', 'list_projects']

Total tools available to the agent: 7


Build the agent over whatever connected 

In [8]:

from langchain.chat_models import init_chat_model


model = init_chat_model(
    model="openrouter/free",
    model_provider="openrouter",
)

In [10]:


assert all_tools, "No tools available to the agent. Check server statuses above."




agent=create_agent(
    model=model,
    tools=all_tools

)
print(f"Agent built with {len(all_tools)} tools:",[t.name for t in all_tools], ". Ready to run.")




Agent built with 7 tools: ['check_showtimes', 'resolve-library-id', 'query-docs', 'log_time', 'get_timesheet', 'get_project_summary', 'list_projects'] . Ready to run.


In [12]:
while True:
    question = input("\nAsk the fleet agent (or 'quit'): ").strip()
    if question.lower() in {"quit", "exit", ""}:
        break
    try: 
        result = await agent.ainvoke({"messages": [("user", question)]})
        for msg in result["messages"]:
            if getattr(msg, "tool_calls", None):
                for tc in msg.tool_calls:
                    print(f"  [tool call] {tc['name']}({tc['args']})")
        print("Agent:", result["messages"][-1].content)
    except Exception as e:
        print(f"Call skipped -- {type(e).__name__}: {e}")

Agent: A LangChain agent is a component within the LangChain framework that leverages language models to make decisions and execute actions based on user input.

## Core Concept:
An agent consists of three key elements:

1. **LLM/Model**: The reasoning engine that determines what action to take
2. **Memory**: The agent's ability to remember previous interactions and tools used
3. **Tools**: The instruments the agent can use to interact with the world (like search APIs, calculators, or other utilities)

## How It Works:
1. The LLM receives the user's request along with the available tools and their descriptions
2. Based on this input, it decides what action to take (which tool to use and with what parameters)
3. The tool executes and returns its output
4. The result is fed back to the model, which can either request another action or provide a final response
5. This process continues until the task is complete

## Example:
If asked "What was the GDP of France in 2020?", an agent would:
